In [ ]:
import pandas as pd

# 1. 準備你的測站字典
station_dict = {
    "Hualien": [
        "TWF1", "TWD", "SHUL", "HWA", "FULB", "EYUL", "EYL", "ETM", 
        "ETLH", "ETL", "ESL", "EHYH", "EHY", "EHP", "EGFH", "EGC"
    ],
    "Taipei": [
        "ZUZH", "TWA", "TAP", "NHY", "ANP", "TWY", "TWS1", "TWB1", 
        "TIPB", "NWR", "NWL", "NWF", "NTS", "NSM", "NHDH", "BAC"
    ]
}

# 將花蓮與台北的測站合併成一個大 List
target_stations = station_dict["Hualien"] + station_dict["Taipei"]

# 2. 讀取原始的 CSV 檔案
df = pd.read_csv('../source/all_metadata.csv') 

# ----------------- 開始執行核心流程 -----------------

# 步驟一：抓取所有不同的 event_id 
unique_event_ids = df['source_event_id'].unique()
filtered_results = []

# 步驟二：依照 event_id 依序去過濾 station_code
for event in unique_event_ids:
    event_data = df[df['source_event_id'] == event]
    
    # 使用合併後的 target_stations 一次性過濾
    target_data = event_data[event_data['station_code'].isin(target_stations)]
    
    # 如果這個事件過濾後有資料，就加入結果中
    if not target_data.empty:
        filtered_results.append(target_data)


# 步驟三：將結果存成一個新的 CSV 檔案
if len(filtered_results) > 0:
    final_df = pd.concat(filtered_results, ignore_index=True)
    
    # 將 'source_event_id' 放到清單的最前面
    cols = final_df.columns.tolist()
    cols.remove('source_event_id')
    cols = ['source_event_id'] + cols
    final_df = final_df[cols]
    
    output_filename = '../source/filtered_hualien_taipei_events.csv'
    final_df.to_csv(output_filename, index=False)
    
    print(f"處理完成！總共處理了 {len(unique_event_ids)} 個獨立事件。")
    print(f"篩選後的結果已儲存為 '{output_filename}'，共包含 {len(final_df)} 筆紀錄。")
else:
    print("這些事件中，沒有任何一個符合花蓮或台北測站的紀錄。")

# 步驟四:# 只要這一行，就能完成分群、找最小值、轉字典的所有動作！
event_earliest_p_arrival_sample = df.groupby('source_event_id')['trace_p_arrival_sample'].min().to_dict()
print(event_earliest_p_arrival_sample)

In [ ]:
"""
批次波形裁切程式
=================
功能：
  1. 讀取 filtered_hualien_taipei_events.csv 中的 source_event_id 與 trace_name
  2. 建立 event_earliest_p_arrival_sample 字典（每個事件中最早的 P 波到達時間）
  3. 用該事件最早P波到達時間計算裁切範圍: [p_arrival - 500, p_arrival + 1500]（共 2000 點）
  4. 從 HDF5 檔案中讀取對應波形並裁切，存成 trace_name.npy

依賴：tool/hdf5_waveform_slicer.py 中的 WaveformExtractor
"""

import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import sys
import time
import pandas as pd
import numpy as np

# 1. 取得上一層資料夾的絕對路徑 (假設你的 notebook 跟 tool 分屬不同子資料夾)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# 2. 將這個上一層路徑加入 Python 的系統搜尋路徑中
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from tool.tool.hdf5_waveform_slicer import WaveformExtractor


def main():
    # ==========================================
    # 路徑設定
    # ==========================================
    csv_path = os.path.join(parent_dir, 'source', 'unfinish_cleaned.csv')
    hdf5_path = '/Volumes/mcnlab2/CWA_processed_data/all.hdf5'
    output_dir = os.path.join(os.getcwd(), 'extracted_waveforms')

    # ==========================================
    # 步驟 1：讀取 CSV
    # ==========================================
    print(f"[*] 讀取 CSV: {csv_path}")
    df = pd.read_csv(csv_path, low_memory=False)
    df['source_event_id'] = df['source_event_id'].astype(str).str.strip()
    # 🟢 確保 trace_name 也是乾淨的字串
    df['trace_name'] = df['trace_name'].astype(str).str.strip()
    print(f"[+] CSV 共 {len(df)} 筆紀錄")

    # ==========================================
    # 步驟 2：建立 event_earliest_p_arrival_sample 字典
    #   key: source_event_id (str)
    #   value: 該事件所有測站中最早的 trace_p_arrival_sample (int)
    # ==========================================
    event_earliest_p_arrival_sample = (
        df.groupby('source_event_id')['trace_p_arrival_sample']
        .min()
        .to_dict()
    )
    print(f"[+] 共建立 {len(event_earliest_p_arrival_sample)} 個事件的最早 P 波到達時間字典")

    # ==========================================
    # 步驟 3：去除重複的 (event_id, trace_name) 組合
    #   🟢 改用 trace_name 來去重，完美對接 extractor
    # ==========================================
    unique_traces = df[['source_event_id', 'trace_name']].drop_duplicates()
    print(f"[+] 去重後共有 {len(unique_traces)} 條 Trace 需要處理")

    # ... (前面的 import 與讀取 CSV 步驟 1~3 完全不變) ...

    # ==========================================
    # 步驟 4 & 5：使用 with 語法安全開啟並批次處理
    # ==========================================
    os.makedirs(output_dir, exist_ok=True)
    success_count = 0
    skip_no_p = 0
    skip_not_found = 0
    total = len(unique_traces)

    print(f"[*] 開始批次提取波形，輸出目錄: {output_dir}")
    print("=" * 60)

    start_time = time.time()

    # 🟢 關鍵保護機制：使用 with 開啟，確保不管發生什麼事，檔案最後都會關閉
    with WaveformExtractor(hdf5_path) as extractor:
        
        for i, (_, row) in enumerate(unique_traces.iterrows()):
            event_id = row['source_event_id']
            trace_name = row['trace_name']

            p_arrival = event_earliest_p_arrival_sample.get(event_id)
            if p_arrival is None or pd.isna(p_arrival):
                skip_no_p += 1
                continue

            p_arrival = int(p_arrival)
            start_idx = p_arrival - 500
            end_idx = p_arrival + 1500
            slice_range = (start_idx, end_idx)

            result = extractor.extract(
                trace_name=trace_name,
                slice_range=slice_range,
                output_dir=output_dir
            )

            if result is not None:
                success_count += 1
                # 改為每 100 筆印一次，因為現在速度會非常快
                if success_count % 100 == 0 or success_count <= 5:
                    print(f"  [{success_count}/{total}] 已儲存: {os.path.basename(result)} ")
            else:
                skip_not_found += 1

    # 離開 with 縮排區塊後，Extractor 的 __exit__ 會自動被觸發並關閉檔案

    # ==========================================
    # 步驟 6：結果統計
    # ==========================================
    total_time = time.time() - start_time
    print("=" * 60)
    print(f"[完成] 批次波形裁切結束！總耗時: {total_time:.2f} 秒")
    print(f"  - 成功儲存: {success_count} 筆")
    print(f"  - 跳過 (無P波資料): {skip_no_p} 筆")
    print(f"  - 跳過 (HDF5找不到): {skip_not_found} 筆")
    print(f"  - 輸出目錄: {output_dir}")

if __name__ == "__main__":
    main()